In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
# !pip install happytransformer
# !pip install wordcloud
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)
from happytransformer import HappyTextToText, TTSettings, TTTrainArgs
import pickle

2024-10-30 14:06:47.605446: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-30 14:06:47.754967: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-10-30 14:06:48.486677: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/lib:/usr/local/nvidia/lib:/usr/local/nvidia/lib64
2024-10-30 14:

In [2]:
df = pd.read_csv('combined_Spanish.csv')
df = df.rename(columns={'text': 'input', 'correct': 'target'})
df['input'] = df['input'].apply(lambda x: f"grammar: {x}")

In [3]:
print(df)

                                                    input  \
0       grammar: It seems not easy to find a job for w...   
1       grammar: N so it is a great success,however it...   
2       grammar: Paris should bear all the blame, Qin ...   
3       grammar: Today is father's day for my family b...   
4       grammar: I like the content of the conversatio...   
...                                                   ...   
100572  grammar: It had been traditionally supposed th...   
100573  grammar: I watched Reading Festival 2010's vid...   
100574             grammar: My senior is going to Europa.   
100575  grammar: Last thursday, I went for a class of ...   
100576  grammar: I was very surprised to find that iMa...   

                                                   target  
0       It doesn't seem easy for women to find jobs in...  
1       N so it is a great success. However it failed ...  
2       Paris should bear all the blame, Qin said, urg...  
3                    Today 

In [4]:
train_ratio = 0.8
train_size = int(train_ratio * len(df))
train_data = df[:train_size]
eval_data = df[train_size:]

train_data.to_csv("train1.csv", index=False)
eval_data.to_csv("eval1.csv", index=False)

In [5]:
happy_tt = HappyTextToText("T5", "t5-base")

args = TTTrainArgs(
    batch_size=4,  # Increase batch size
    learning_rate=5e-5,  # Lower learning rate for more stable convergence
    max_input_length=32,  # Adjust if necessary for longer sentences
    max_output_length=32,  # Adjust for longer output sentences
    num_train_epochs=1,  # Increase the number of training epochs
    eval_ratio=0.1  # Use 10% of the data for evaluation during training
)

happy_tt.train("train1.csv", args=args)

10/30/2024 14:06:51 - INFO - happytransformer.happy_transformer -   Using device: cpu


Generating train split: 0 examples [00:00, ? examples/s]

10/30/2024 14:06:52 - INFO - happytransformer.happy_transformer -   Tokenizing training data...


Map:   0%|          | 0/72414 [00:00<?, ? examples/s]

Map:   0%|          | 0/8047 [00:00<?, ? examples/s]

10/30/2024 14:07:01 - INFO - happytransformer.happy_transformer -   Moving model to cpu
/home/jovyan/.local/lib/python3.8/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Step,Training Loss,Validation Loss
1,1.691700,2.254286
1811,1.452400,1.312754
3622,1.378800,1.278471
5433,1.375800,1.259420
7244,1.336700,1.249604
9055,1.347500,1.237382
10866,1.309100,1.228453
12677,1.313000,1.221762
14488,1.306500,1.218222
16299,1.298200,1.216979


In [6]:
with open('model1', 'wb') as file:
    pickle.dump(happy_tt, file)

In [7]:
with open('model1', 'rb') as file:
    happy_tt = pickle.load(file)

beam_settings = TTSettings(num_beams=5, min_length=1, max_length=50)
eval1 = pd.read_csv('eval1.csv')
end = 100

learner_sentences = eval1.iloc[:end, 0]
target = eval1.iloc[:end, 1]

corrected = []
for sentence in learner_sentences:
    corrected_sentence = happy_tt.generate_text(sentence, args=beam_settings).text
    corrected.append(corrected_sentence)

c = 0
nc = 0
s = 0

for i in range(len(corrected)):
    ls = learner_sentences[i].replace("grammar: ", "")
    print(ls)
    print(corrected[i])
    print(target[i])
    if (corrected[i] == target[i]):
        c = c+1
    else:
        if (corrected[i] == ls):
            nc = nc+1
        else:
            s = s+1

print(c, nc, s)

10/30/2024 23:05:47 - INFO - happytransformer.happy_transformer -   Initializing a pipeline


From February, I re-rented a parking space at more near my apartment because the space I had rented so far was closed.
From February, I re-rented a parking space near my apartment because the space I had rented so far was closed.
Last February, I rented a parking space nearer to my apartment, because the space I had been renting before was closed.
I like the commute time that I listen to music or radio at.
I like the commute time when I listen to music or radio.
I like using the time while I am commuting to listen to the radio, especially to music.
In Takarazuka performance all parts are played by female and so are man's role.
In Takarazuka performance all parts are played by females and so is the man's role.
In Takarazuka performances all the parts are played bywomen even male roles.
In fact, I have the other lang-8 account.
In fact, I have another lang-8 account.
In fact, I have another ang-8 account.
But recently I don't write a diary, so I forget my password.
But recently I haven't